In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # JobFlow AI — Tabelas Transacionais do App em Delta
# MAGIC
# MAGIC Como Lakebase não aparece no workspace atual, este notebook cria
# MAGIC uma camada transacional temporária usando Delta Tables.
# MAGIC
# MAGIC Essas tabelas simulam a estrutura que depois poderá ser migrada
# MAGIC para Lakebase/PostgreSQL.

# COMMAND ----------

from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql import types as T

# COMMAND ----------

CATALOG = "workspace"
SCHEMA = "jobflow_ai"

GOLD_JOBS_TABLE = f"{CATALOG}.{SCHEMA}.gold_job_postings"

APP_USERS_TABLE = f"{CATALOG}.{SCHEMA}.app_users"
APP_PROFILES_TABLE = f"{CATALOG}.{SCHEMA}.app_profiles"
APP_SKILLS_TABLE = f"{CATALOG}.{SCHEMA}.app_skills"
APP_JOB_POSTINGS_TABLE = f"{CATALOG}.{SCHEMA}.app_job_postings"
APP_SAVED_JOBS_TABLE = f"{CATALOG}.{SCHEMA}.app_saved_jobs"
APP_APPLICATIONS_TABLE = f"{CATALOG}.{SCHEMA}.app_applications"
APP_INTERVIEW_NOTES_TABLE = f"{CATALOG}.{SCHEMA}.app_interview_notes"
APP_CONTACTS_TABLE = f"{CATALOG}.{SCHEMA}.app_contacts"

DEMO_USER_ID = "demo_user_001"
DEMO_PROFILE_ID = "demo_profile_data_engineer"

print("=" * 70)
print("JOBFLOW AI — TABELAS TRANSACIONAIS DO APP")
print("=" * 70)
print(f"Gold jobs table: {GOLD_JOBS_TABLE}")
print(f"Horário UTC: {datetime.now(timezone.utc).isoformat()}")
print("=" * 70)

# COMMAND ----------

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Criar tabela app_users

# COMMAND ----------

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {APP_USERS_TABLE} (
        user_id STRING NOT NULL,
        email STRING,
        display_name STRING,
        locale STRING,
        created_at TIMESTAMP,
        updated_at TIMESTAMP
    )
    USING DELTA
    """
)

print(f"OK: {APP_USERS_TABLE}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Criar tabela app_profiles

# COMMAND ----------

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {APP_PROFILES_TABLE} (
        profile_id STRING NOT NULL,
        user_id STRING NOT NULL,
        headline STRING,
        target_roles ARRAY<STRING>,
        preferred_location STRING,
        remote_preference STRING,
        min_salary BIGINT,
        salary_currency STRING,
        seniority_target STRING,
        resume_text STRING,
        created_at TIMESTAMP,
        updated_at TIMESTAMP
    )
    USING DELTA
    """
)

print(f"OK: {APP_PROFILES_TABLE}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Criar tabela app_skills

# COMMAND ----------

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {APP_SKILLS_TABLE} (
        skill_id STRING NOT NULL,
        skill_name STRING NOT NULL,
        skill_category STRING,
        aliases ARRAY<STRING>,
        created_at TIMESTAMP,
        updated_at TIMESTAMP
    )
    USING DELTA
    """
)

print(f"OK: {APP_SKILLS_TABLE}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Criar tabela app_job_postings

# COMMAND ----------

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {APP_JOB_POSTINGS_TABLE} (
        job_id STRING NOT NULL,
        source_system STRING,
        source_job_id STRING,
        source_job_key STRING,
        job_title STRING,
        company_name STRING,
        job_location STRING,
        remote_type STRING,
        employment_type STRING,
        published_at TIMESTAMP,
        tags_text STRING,
        job_description STRING,
        salary_min BIGINT,
        salary_max BIGINT,
        salary_midpoint DOUBLE,
        salary_currency STRING,
        is_active BOOLEAN,
        data_quality_score DOUBLE,
        apply_url STRING,
        job_url STRING,
        created_at TIMESTAMP,
        updated_at TIMESTAMP
    )
    USING DELTA
    """
)

print(f"OK: {APP_JOB_POSTINGS_TABLE}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Criar tabela app_saved_jobs

# COMMAND ----------

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {APP_SAVED_JOBS_TABLE} (
        saved_job_id STRING NOT NULL,
        user_id STRING NOT NULL,
        job_id STRING NOT NULL,
        priority STRING,
        notes STRING,
        saved_at TIMESTAMP,
        updated_at TIMESTAMP
    )
    USING DELTA
    """
)

print(f"OK: {APP_SAVED_JOBS_TABLE}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Criar tabela app_applications

# COMMAND ----------

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {APP_APPLICATIONS_TABLE} (
        application_id STRING NOT NULL,
        user_id STRING NOT NULL,
        job_id STRING NOT NULL,
        stage STRING,
        status STRING,
        applied_at TIMESTAMP,
        last_activity_at TIMESTAMP,
        next_followup_at TIMESTAMP,
        notes STRING,
        created_at TIMESTAMP,
        updated_at TIMESTAMP
    )
    USING DELTA
    """
)

print(f"OK: {APP_APPLICATIONS_TABLE}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 7. Criar tabela app_interview_notes

# COMMAND ----------

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {APP_INTERVIEW_NOTES_TABLE} (
        note_id STRING NOT NULL,
        application_id STRING NOT NULL,
        user_id STRING NOT NULL,
        job_id STRING,
        round_type STRING,
        note_text STRING,
        created_at TIMESTAMP,
        updated_at TIMESTAMP
    )
    USING DELTA
    """
)

print(f"OK: {APP_INTERVIEW_NOTES_TABLE}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 8. Criar tabela app_contacts

# COMMAND ----------

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {APP_CONTACTS_TABLE} (
        contact_id STRING NOT NULL,
        user_id STRING NOT NULL,
        company_name STRING,
        contact_name STRING,
        title STRING,
        email STRING,
        linkedin_url STRING,
        last_contact_at TIMESTAMP,
        notes STRING,
        created_at TIMESTAMP,
        updated_at TIMESTAMP
    )
    USING DELTA
    """
)

print(f"OK: {APP_CONTACTS_TABLE}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 9. Inserir usuário e perfil demo

# COMMAND ----------

now_expr = "current_timestamp()"

spark.sql(
    f"""
    DELETE FROM {APP_USERS_TABLE}
    WHERE user_id = '{DEMO_USER_ID}'
    """
)

spark.sql(
    f"""
    INSERT INTO {APP_USERS_TABLE}
    SELECT
        '{DEMO_USER_ID}' AS user_id,
        'demo.user@example.com' AS email,
        'Demo Data Engineer' AS display_name,
        'pt-BR' AS locale,
        {now_expr} AS created_at,
        {now_expr} AS updated_at
    """
)

spark.sql(
    f"""
    DELETE FROM {APP_PROFILES_TABLE}
    WHERE profile_id = '{DEMO_PROFILE_ID}'
    """
)

spark.sql(
    f"""
    INSERT INTO {APP_PROFILES_TABLE}
    SELECT
        '{DEMO_PROFILE_ID}' AS profile_id,
        '{DEMO_USER_ID}' AS user_id,
        'Data Engineer buscando vagas remotas com Python, SQL, Spark e Databricks' AS headline,
        array('data engineer', 'analytics engineer', 'backend engineer') AS target_roles,
        'remote' AS preferred_location,
        'remote' AS remote_preference,
        70000 AS min_salary,
        'USD' AS salary_currency,
        'mid' AS seniority_target,
        'Perfil demo com experiência em Python, SQL, Spark, Databricks, ETL, APIs e analytics.' AS resume_text,
        {now_expr} AS created_at,
        {now_expr} AS updated_at
    """
)

print("OK: usuário e perfil demo inseridos.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 10. Publicar skills do catálogo para app_skills

# COMMAND ----------

skills_source_df = spark.table(f"{CATALOG}.{SCHEMA}.skills_catalog")

app_skills_df = (
    skills_source_df
    .select(
        "skill_id",
        "skill_name",
        "skill_category",
        "aliases",
        F.current_timestamp().alias("created_at"),
        F.current_timestamp().alias("updated_at"),
    )
)

(
    app_skills_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(APP_SKILLS_TABLE)
)

print(f"OK: skills publicadas em {APP_SKILLS_TABLE}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 11. Publicar vagas Gold para app_job_postings

# COMMAND ----------

gold_jobs_df = spark.table(GOLD_JOBS_TABLE)

app_jobs_df = (
    gold_jobs_df
    .select(
        "job_id",
        "source_system",
        "source_job_id",
        "source_job_key",
        "job_title",
        "company_name",
        "job_location",
        "remote_type",
        "employment_type",
        "published_at",
        "tags_text",
        "job_description",
        "salary_min",
        "salary_max",
        "salary_midpoint",
        "salary_currency",
        "is_active",
        "data_quality_score",
        "apply_url",
        "job_url",
        F.current_timestamp().alias("created_at"),
        F.current_timestamp().alias("updated_at"),
    )
)

(
    app_jobs_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(APP_JOB_POSTINGS_TABLE)
)

print(f"OK: vagas publicadas em {APP_JOB_POSTINGS_TABLE}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 12. Criar exemplos de ações do usuário

# COMMAND ----------

top_jobs_df = (
    spark.table(f"{CATALOG}.{SCHEMA}.gold_job_match_scores")
    .orderBy(F.col("match_score").desc())
    .limit(3)
    .select("job_id", "job_title", "company_name", "match_score")
)

display(top_jobs_df)

top_jobs = top_jobs_df.collect()

if len(top_jobs) >= 2:
    first_job_id = top_jobs[0]["job_id"]
    second_job_id = top_jobs[1]["job_id"]

    saved_rows = [
        {
            "saved_job_id": f"{DEMO_USER_ID}_{first_job_id}",
            "user_id": DEMO_USER_ID,
            "job_id": first_job_id,
            "priority": "high",
            "notes": "Vaga salva automaticamente para demonstração.",
        },
        {
            "saved_job_id": f"{DEMO_USER_ID}_{second_job_id}",
            "user_id": DEMO_USER_ID,
            "job_id": second_job_id,
            "priority": "medium",
            "notes": "Segunda vaga recomendada para revisão.",
        },
    ]

    saved_schema = T.StructType(
        [
            T.StructField("saved_job_id", T.StringType(), False),
            T.StructField("user_id", T.StringType(), False),
            T.StructField("job_id", T.StringType(), False),
            T.StructField("priority", T.StringType(), True),
            T.StructField("notes", T.StringType(), True),
        ]
    )

    saved_df = (
        spark.createDataFrame(saved_rows, saved_schema)
        .withColumn("saved_at", F.current_timestamp())
        .withColumn("updated_at", F.current_timestamp())
    )

    spark.sql(
        f"""
        DELETE FROM {APP_SAVED_JOBS_TABLE}
        WHERE user_id = '{DEMO_USER_ID}'
        """
    )

    saved_df.write.mode("append").saveAsTable(APP_SAVED_JOBS_TABLE)

    application_rows = [
        {
            "application_id": f"{DEMO_USER_ID}_{first_job_id}_application",
            "user_id": DEMO_USER_ID,
            "job_id": first_job_id,
            "stage": "saved",
            "status": "active",
            "notes": "Aplicação demo criada a partir da melhor recomendação.",
        }
    ]

    application_schema = T.StructType(
        [
            T.StructField("application_id", T.StringType(), False),
            T.StructField("user_id", T.StringType(), False),
            T.StructField("job_id", T.StringType(), False),
            T.StructField("stage", T.StringType(), True),
            T.StructField("status", T.StringType(), True),
            T.StructField("notes", T.StringType(), True),
        ]
    )

    applications_df = (
        spark.createDataFrame(application_rows, application_schema)
        .withColumn("applied_at", F.lit(None).cast("timestamp"))
        .withColumn("last_activity_at", F.current_timestamp())
        .withColumn("next_followup_at", F.expr("current_timestamp() + INTERVAL 7 DAYS"))
        .withColumn("created_at", F.current_timestamp())
        .withColumn("updated_at", F.current_timestamp())
    )

    spark.sql(
        f"""
        DELETE FROM {APP_APPLICATIONS_TABLE}
        WHERE user_id = '{DEMO_USER_ID}'
        """
    )

    applications_df.write.mode("append").saveAsTable(APP_APPLICATIONS_TABLE)

    print("OK: vagas salvas e aplicação demo criadas.")
else:
    print("Não há vagas suficientes para criar exemplos.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 13. Validações finais

# COMMAND ----------

tables_to_validate = [
    APP_USERS_TABLE,
    APP_PROFILES_TABLE,
    APP_SKILLS_TABLE,
    APP_JOB_POSTINGS_TABLE,
    APP_SAVED_JOBS_TABLE,
    APP_APPLICATIONS_TABLE,
    APP_INTERVIEW_NOTES_TABLE,
    APP_CONTACTS_TABLE,
]

validation_rows = []

for table_name in tables_to_validate:
    count_value = spark.table(table_name).count()
    validation_rows.append((table_name, count_value))

validation_df = spark.createDataFrame(
    validation_rows,
    ["table_name", "records"],
)

display(validation_df)

display(
    spark.sql(
        f"""
        SELECT
            s.saved_job_id,
            s.priority,
            j.job_title,
            j.company_name,
            j.remote_type
        FROM {APP_SAVED_JOBS_TABLE} s
        INNER JOIN {APP_JOB_POSTINGS_TABLE} j
            ON s.job_id = j.job_id
        WHERE s.user_id = '{DEMO_USER_ID}'
        """
    )
)

display(
    spark.sql(
        f"""
        SELECT
            a.application_id,
            a.stage,
            a.status,
            a.next_followup_at,
            j.job_title,
            j.company_name
        FROM {APP_APPLICATIONS_TABLE} a
        INNER JOIN {APP_JOB_POSTINGS_TABLE} j
            ON a.job_id = j.job_id
        WHERE a.user_id = '{DEMO_USER_ID}'
        """
    )
)

# COMMAND ----------

print()
print("=" * 70)
print("RESULTADO: TABELAS TRANSACIONAIS DELTA CONCLUÍDAS")
print("=" * 70)
print(f"users table: {APP_USERS_TABLE}")
print(f"profiles table: {APP_PROFILES_TABLE}")
print(f"skills table: {APP_SKILLS_TABLE}")
print(f"job postings table: {APP_JOB_POSTINGS_TABLE}")
print(f"saved jobs table: {APP_SAVED_JOBS_TABLE}")
print(f"applications table: {APP_APPLICATIONS_TABLE}")
print(f"interview notes table: {APP_INTERVIEW_NOTES_TABLE}")
print(f"contacts table: {APP_CONTACTS_TABLE}")
print("=" * 70)